In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese Versionen oder neuere zu verwenden.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Neben dem [Visualisieren von Anweisungen in einem Circuit](/guides/visualize-circuits) möchtest du vielleicht auch das Scheduling eines Circuits visualisieren, indem du die Qiskit-Methode [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) verwendest. Diese Visualisierung kann dir dabei helfen, beispielsweise Leerlaufzeiten auf Qubits schnell zu erkennen. Allerdings liefert diese Methode für dynamische Circuits keine genauen Ergebnisse. Um das Scheduling dynamischer Circuits zu visualisieren, verwende die Methode `draw_circuit_schedule_timing`, wie im Abschnitt [Qiskit Runtime-Unterstützung](#qr-support) beschrieben.

## Beispiele

Um ein geplantes Circuit-Programm zu visualisieren, kannst du diese Funktion mit einer Reihe von Steuerargumenten aufrufen. Der Großteil des Erscheinungsbilds des Ausgabebilds lässt sich über ein Stylesheet anpassen, was jedoch nicht zwingend erforderlich ist.

### Mit dem Standard-Stylesheet zeichnen

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Mit einem für das Programm-Debugging geeigneten Stylesheet zeichnen

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Du kannst benutzerdefinierte Generator- oder Layout-Funktionen erstellen und ein vorhandenes Stylesheet mit diesen Funktionen aktualisieren. Auf diese Weise kannst du den Großteil des Erscheinungsbilds des Ausgabebilds steuern, ohne die Codebasis des geplanten Circuit-Zeichners zu ändern. Weitere Beispiele findest du in der API-Referenz für [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer).
<span id="qr-support"></span>

## Qiskit Runtime-Unterstützung

Während der in Qiskit integrierte Timeline-Drawer für statische Circuits nützlich ist, spiegelt er das Timing von [dynamischen Circuits](/guides/classical-feedforward-and-control-flow) möglicherweise nicht korrekt wider, da implizite Operationen wie Broadcasting und Branch-Bestimmung auftreten. Als Teil der Unterstützung für dynamische Circuits gibt Qiskit Runtime auf Anfrage die genauen Circuit-Timing-Informationen in den Job-Ergebnissen zurück.

> **Note:** - Dies ist eine experimentelle Funktion. Sie befindet sich im Vorschau-Status und kann daher geändert werden.
> - Diese Funktion gilt nur für Qiskit Runtime Sampler-Jobs.
> - Obwohl die gesamte Circuit-Zeit in den „compilation"-Metadaten zurückgegeben wird, ist dies NICHT die für die Abrechnung verwendete Zeit (Quantenzeit).

### Timing-Datenabruf aktivieren

Um den Timing-Datenabruf zu aktivieren, setze das experimentelle Flag `scheduler_timing` beim Ausführen des Primitive-Jobs auf `True`.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Auf die Circuit-Timing-Daten zugreifen
Wenn angefordert, werden die Circuit-Timing-Daten für jeden PUB in den Job-Ergebnis-Metadaten zurückgegeben, unter `["compilation"]["scheduler_timing"]["timing"]`. Dieses Feld enthält die rohen Timing-Informationen. Um die Timing-Informationen anzuzeigen, verwende das integrierte Visualisierungswerkzeug, wie im Abschnitt [Timings visualisieren](#visualize-timings) beschrieben.

Verwende den folgenden Code, um auf die Circuit-Timing-Daten für den ersten PUB zuzugreifen:

In [4]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Die rohen Timing-Daten verstehen
Das Visualisieren der Circuit-Timing-Daten mit der Methode `draw_circuit_schedule_timing` ist zwar der häufigste Anwendungsfall, aber es kann nützlich sein, die Struktur der zurückgegebenen rohen Timing-Daten zu verstehen. Dies kann dir beispielsweise helfen, Informationen programmatisch zu extrahieren.

Die in `["compilation"]["scheduler_timing"]["timing"]` zurückgegebenen Timing-Daten sind eine Liste von Zeichenketten. Jede Zeichenkette repräsentiert eine einzelne Anweisung auf einem bestimmten Kanal und ist durch Kommas in die folgenden Datentypen unterteilt:

- `Branch` – Bestimmt, ob sich die Anweisung in einem Control-Flow (then / else) oder einem Hauptzweig befindet.
- `Instruction` – Das Gate und das Qubit, auf dem es angewendet wird.
- `Channel` – Der Kanal, dem die Anweisung zugewiesen wird. Es kann einer der folgenden sein:
      - `Qubit x` – Der Drive-Kanal für Qubit _x_.
      - `AWGRx_y` (arbitrary waveform generator readout) – Wird von Auslesekanälen verwendet, um zu kommunizieren, wann Qubits gemessen werden. Die Argumente _x_ und _y_ entsprechen der Ausleseinstrument-ID bzw. der Qubit-Nummer.
- `T0` – Die Startzeit der Anweisung innerhalb des gesamten Schedules.
- `Duration` – Die Dauer der Anweisung in Einheiten von _dt_ Sekunden, wobei 1 dt = 1 Scheduling-Zyklus. Den `dt`-Wert eines Backends kannst du über [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt) ermitteln.
- `Pulse` – Der Typ der verwendeten Puls-Operation.

Beispiel:

In [5]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Timings visualisieren
Mit `qiskit-ibm-runtime` v0.43.0 oder höher kannst du Circuit-Timings visualisieren. Um die Timings zu visualisieren, musst du zunächst die Ergebnis-Metadaten mit der Methode [`draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) in `fig` umwandeln. Diese Methode gibt eine `plotly`-Abbildung zurück, die du direkt anzeigen, in einer Datei speichern oder beides tun kannst. Weitere Informationen zu den zu verwendenden `plotly`-Befehlen findest du unter [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) und [`fig.write_image("<path.format>")`.](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html)

In [6]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d5kk3cn853es738e01dg (DONE)


![Beim Überfahren der Ausgabe mit der Maus werden Informationen wie Start, Ende und Dauer angezeigt.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Beispiel einer generierten Abbildung')
#### Die generierte Abbildung verstehen
Das Bild der Circuit-Timing-Daten, das von `draw_circuit_schedule_timing` ausgegeben wird, vermittelt folgende Informationen:

- Die X-Achse ist die Zeit in Einheiten von _dt_ Sekunden, wobei 1 dt = 1 Scheduling-Zyklus. Den `dt`-Wert eines Backends kannst du über [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt) ermitteln.
- Die Y-Achse ist der Kanal (stelle dir Kanäle als Instrumente vor, die Pulse aussenden).
    - `Receive channel` – Der einzige Kanal, der kein eigenständiges Instrument ist. Es handelt sich um eine Anweisung, die auf allen Kanälen ausgeführt wird, die zu einem bestimmten Zeitpunkt an einem Kommunikationsvorgang mit dem Hub beteiligt sind.
    - `Qubit x` – Der Drive-Kanal für Qubit x.
    - `AWGRx_y` (arbitrary waveform generator readout) – Wird von Auslesekanälen verwendet, um zu kommunizieren, wann Qubits gemessen werden. Die Argumente _x_ und _y_ entsprechen der Ausleseinstrument-ID bzw. der Qubit-Nummer.
    - `Hub` – Steuert das Broadcasting.

Zusätzlich hat jede Anweisung das Format *X_Y*, wobei *X* der Name der Anweisung und *Y* der Pulstyp ist. Ein `play`-Typ wendet Steuerpulse an, und ein `capture` zeichnet den Zustand des Qubits auf. Du kannst auch mit der Maus über jede Anweisung fahren, um weitere Details zu erhalten. Die vorherige Abbildung zeigt beispielsweise einen Steuerpuls für das X-Gate, das auf Qubit 10 bei 1161 dt angewendet wird.
### End-to-End-Beispiel
Dieses Beispiel zeigt dir, wie du die Option aktivierst, die Circuit-Timing-Informationen aus den Metadaten abrufst und sie als Bild darstellst.

Richte zunächst die Umgebung ein, definiere die Circuits und konvertiere sie in ISA-Circuits, und definiere und führe die Jobs aus.

In [7]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,929,0,shift_phase\nmain,sx_0,Qubit 0,929,8,play\nmain,sx_0,Qubit 0,933,0,shift_phase\nmain,rz_0,Qubit 0,937,0,shift_phase\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 1,937,0,barrier\nmain,barrier,Qubit 2,937,0,barrier\nmain,barrier,Qubit 3,937,0,barrier\nmain,barrier,Qubit 4,937,0,barrier\nmain,barrier,Qubit 5,937,0,barrier\nmain,barrier,Qubit 6,937,0,barrier\nmain,barrier,Qubit 7,937,0,barrier\nmain,barrier,Qubit 8,937,0,barrier\nmain,barrier,Qubit 9,937,0,barrier\nmain,barrier,Qubit 10,937,0,barrier\nmain,barrier,Qubit 11,937,0,barrier\nmain,barrier,Qubit 12,937,0,barrier\nmain,barrier,Qubit 13,937,0,barrier\nmain,barrier,Qubit 14,937,0,barrier\nmain,barrier,Qubit 15,937,0,barrier\nmain,barrier,Qubit 16,937,0,barrier\nmain,barrier,Qubit 17,937,0,barrier\nmain,barrier,Qubit 18,937,0,barrier\nmain,barrier,Qubit 19,937,0,barrier\nmain,barrier,Qubit 20,937,0,barrier\nmain,barrier,Qubit 21,937,0,barrier\nmain,barrier,Qubit

Finally, you can visualize and save the timing:

In [8]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Als nächstes rufst du das Circuit-Schedule-Timing ab: